In [1]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'Pi_coef'
_AUGMENTED_EQUATION = 'OutGap'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.01
_MC_SAMPLES = 100_000
_MC_ALPHA = 0.05
_MC_SUMMARY_ONLY = True
_MC_INCLUDE_BY_PREDICTOR = False
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83  -0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [3]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [4]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [5]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
    summary_only=_MC_SUMMARY_ONLY,
    include_by_predictor=_MC_INCLUDE_BY_PREDICTOR,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [6]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: OutGap
Augmented coefficient: Pi_coef
Monte Carlo replications: 100000
Noise Covariance:
 [[0.121 0.    0.   ]
 [0.    0.168 0.   ]
 [0.    0.    0.008]]


In [7]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
mc_reference["lb_summary"].round(3)

Monte Carlo Ljung-Box summary across 100000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,15.730,0.003,0.021,0.000,100000,98575,0.986,0.000,0.985,0.986
1,Infl,23.221,0.000,0.025,0.000,100000,99962,1.000,0.000,0.999,1.000
2,Rate,5.993,0.098,0.014,0.001,100000,62550,0.626,0.002,0.622,0.628


In [8]:
display(mc_reference["moment_specification_test_summary"].round(3))

,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.120,2.632,0.562,0.000,0.008,0.001,100000,4594,0.046,0.001,0.045,0.047,3.0,200,4
1,cov_identity,14.327,600.207,0.000,0.005,0.716,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


In [9]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-3.071,-0.117,1.855,-1.675,0.207,0.018,0.006,0.0,0.001,0.003,0.001,0.0,100000,38034,0.380,0.002,0.377,0.383
1,OutGap,x,-0.176,-0.051,0.236,-0.724,0.433,0.007,0.001,0.0,0.000,0.003,0.001,0.0,100000,9832,0.098,0.001,0.096,0.100
2,OutGap,r,-0.514,-0.016,2.047,-0.233,0.506,0.005,0.006,0.0,0.001,0.003,0.001,0.0,100000,4554,0.046,0.001,0.044,0.047
3,Infl,Pi,-0.429,-0.012,2.144,-0.177,0.491,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,5542,0.055,0.001,0.054,0.057
4,Infl,x,0.024,0.006,0.272,0.091,0.493,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5428,0.054,0.001,0.053,0.056
5,Infl,r,-0.431,-0.011,2.350,-0.162,0.492,0.005,0.008,0.0,0.001,0.003,0.001,0.0,100000,5603,0.056,0.001,0.055,0.057
6,Rate,Pi,-0.019,-0.005,0.398,-0.064,0.499,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5037,0.050,0.001,0.049,0.052
7,Rate,x,0.009,0.013,0.050,0.181,0.493,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5499,0.055,0.001,0.054,0.056
8,Rate,r,-0.148,-0.022,0.436,-0.318,0.483,0.006,0.001,0.0,0.000,0.003,0.001,0.0,100000,6346,0.063,0.001,0.062,0.065


In [10]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-4.175,-0.337,0.821,-5.072,0.000,0.117,0.003,0.0,0.000,0.003,0.000,0.0,100000,99983,1.000,0.000,1.000,1.000
1,OutGap,x,-0.488,-0.318,0.102,-4.746,0.000,0.104,0.000,0.0,0.000,0.003,0.000,0.0,100000,99927,0.999,0.000,0.999,0.999
0,OutGap,r,1.431,0.051,1.927,0.728,0.473,0.005,0.004,0.0,0.001,0.002,0.001,0.0,100000,3677,0.037,0.001,0.036,0.038
5,Infl,Pi,-0.094,-0.005,1.001,-0.074,0.501,0.005,0.003,0.0,0.000,0.003,0.001,0.0,100000,4939,0.049,0.001,0.048,0.051
4,Infl,x,-0.005,-0.001,0.124,-0.012,0.502,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,4810,0.048,0.001,0.047,0.049
3,Infl,r,-0.303,-0.009,2.214,-0.127,0.497,0.005,0.007,0.0,0.001,0.003,0.001,0.0,100000,5180,0.052,0.001,0.050,0.053
8,Rate,Pi,0.035,0.012,0.186,0.172,0.499,0.005,0.001,0.0,0.000,0.003,0.001,0.0,100000,5071,0.051,0.001,0.049,0.052
7,Rate,x,0.007,0.020,0.023,0.280,0.492,0.005,0.000,0.0,0.000,0.003,0.001,0.0,100000,5599,0.056,0.001,0.055,0.057
6,Rate,r,-0.172,-0.027,0.411,-0.387,0.480,0.006,0.001,0.0,0.000,0.003,0.001,0.0,100000,6551,0.066,0.001,0.064,0.067


In [22]:
print("Innovation decomposition orthogonal summary:")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"])

Innovation decomposition orthogonal summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.804291,-4.875475,-3.071183,-3.071183,-7.356874e-19,6.217493e-16,1.754605e-15,0.003506,0.002625,0.005671,0.005671,2.587632e-18,1.682285e-18,8.814694e-19
1,OutGap,x,0.009490,-0.185277,-0.175787,-0.175787,-2.678406e-20,5.422907e-17,1.754605e-15,0.000445,0.000349,0.000734,0.000734,2.298597e-19,1.530596e-19,8.814694e-19
2,OutGap,r,-0.217378,-0.297121,-0.514499,-0.514499,7.439188e-19,4.444882e-16,1.754605e-15,0.003869,0.002857,0.006272,0.006272,1.858892e-18,1.216456e-18,8.814694e-19
3,Infl,Pi,-0.014364,-0.414505,-0.428870,-0.428870,-1.833348e-17,5.178949e-16,1.754605e-15,0.000711,0.006909,0.006937,0.006937,2.159022e-18,1.408033e-18,8.814694e-19
4,Infl,x,0.001657,0.022379,0.024036,0.024036,-6.198602e-20,6.556561e-17,1.754605e-15,0.000090,0.000878,0.000882,0.000882,2.722303e-19,1.764099e-19,8.814694e-19
5,Infl,r,-0.005161,-0.426193,-0.431355,-0.431355,-8.014395e-20,5.653873e-16,1.754605e-15,0.000781,0.007625,0.007655,0.007655,2.362883e-18,1.544848e-18,8.814694e-19
6,Rate,Pi,0.000285,-0.019076,-0.018791,-0.018791,2.229047e-19,1.917125e-16,1.754605e-15,0.000153,0.001266,0.001267,0.001267,7.685657e-19,4.723910e-19,8.814694e-19
7,Rate,x,-0.000059,0.009250,0.009191,0.009191,1.467978e-19,2.424790e-17,1.754605e-15,0.000019,0.000162,0.000162,0.000162,9.717663e-20,5.969795e-20,8.814694e-19
8,Rate,r,-0.001967,-0.145837,-0.147804,-0.147804,-2.297934e-19,2.179311e-16,1.754605e-15,0.000167,0.001417,0.001415,0.001415,8.752162e-19,5.394994e-19,8.814694e-19


In [23]:
print("Innovation decomposition raw summary:")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"])

Innovation decomposition raw summary:


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.953432e+00,-6.128556,-4.175123,-4.175123,5.651035e-19,6.096579e-16,1.754605e-15,0.001596,0.001101,0.002569,0.002569,2.597737e-18,1.741085e-18,8.814694e-19
1,OutGap,x,2.141854e-01,-0.702094,-0.487909,-0.487909,-2.230854e-19,7.255065e-17,1.754605e-15,0.000201,0.000196,0.000336,0.000336,3.080564e-19,2.055790e-19,8.814694e-19
2,OutGap,r,-8.626923e-01,2.293899,1.431207,1.431207,1.118279e-18,4.658734e-16,1.754605e-15,0.004624,0.003565,0.004391,0.004391,1.941779e-18,1.264957e-18,8.814694e-19
3,Infl,Pi,-1.073192e-03,-0.092669,-0.093742,-0.093742,-2.042954e-17,2.412081e-16,1.754605e-15,0.000331,0.003141,0.003154,0.003154,9.938070e-19,6.403178e-19,8.814694e-19
4,Infl,x,6.689282e-05,-0.004599,-0.004532,-0.004532,-1.981549e-18,2.968671e-17,1.754605e-15,0.000041,0.000388,0.000390,0.000390,1.227250e-19,7.929431e-20,8.814694e-19
5,Infl,r,-5.190615e-03,-0.298121,-0.303312,-0.303312,5.605551e-18,5.304221e-16,1.754605e-15,0.000733,0.007042,0.007066,0.007066,2.208644e-18,1.436983e-18,8.814694e-19
6,Rate,Pi,-6.304164e-06,0.035330,0.035324,0.035324,3.618089e-19,8.941528e-17,1.754605e-15,0.000071,0.000580,0.000582,0.000582,3.574350e-19,2.186535e-19,8.814694e-19
7,Rate,x,-2.573347e-07,0.006961,0.006961,0.006961,6.327108e-20,1.109183e-17,1.754605e-15,0.000009,0.000072,0.000072,0.000072,4.437416e-20,2.718101e-20,8.814694e-19
8,Rate,r,-1.242872e-03,-0.170461,-0.171704,-0.171704,-4.537867e-19,2.059757e-16,1.754605e-15,0.000157,0.001320,0.001319,0.001319,8.255753e-19,5.072598e-19,8.814694e-19


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw.


In [11]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
        summary_only=_MC_SUMMARY_ONLY,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model


### Marginal LR Test Conditional on $\theta_0$


In [12]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,1.959,-2192.139,-918.748,2546.782,0.0,0.0,0.543,0.082,1.024,0.0,100000,100000,1.0,0.0,1.0,1.0


In [13]:
res_mle

OptimizationResult(kind='mle', x=array([1.86375864]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(1.8637586362309642), 'x_coef': np.float64(0.0), 'r_coef': np.float64(0.0)}, success=True, message='CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', fun=np.float64(872.437433474283), loglik=np.float64(-872.437433474283), logprior=np.float64(0.0), logpost=np.float64(-872.437433474283), nfev=8, nit=3, raw=  message: CONVERGENCE: NORM OF P

## Serial Autocorrelation Tests for the Augmented Model


In [14]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,1.063,0.489,0.005,0.001,100000,5741,0.057,0.001,0.056,0.059
1,Infl,12.282,0.013,0.020,0.000,100000,93745,0.937,0.001,0.936,0.939
2,Rate,2.258,0.334,0.008,0.001,100000,20399,0.204,0.001,0.202,0.206


In [15]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.120,2.632,0.562,0.000,0.008,0.001,100000,4594,0.046,0.001,0.045,0.047,3.0,200,4
1,cov_identity,14.327,600.207,0.000,0.005,0.716,0.000,100000,100000,1.000,0.000,1.000,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.113,3.070,0.498,0.000,0.008,0.001,100000,5878,0.059,0.001,0.057,0.060,3.0,200,4
1,cov_identity,0.991,31.713,0.002,0.001,0.031,0.000,100000,99486,0.995,0.000,0.994,0.995,6.0,200,4


Reference-minus-augmented moment distance comparison:


""
